# Structured Output Quickstart

This notebook shows how Agentic Forge wires together prompts, schemas, and model managers into named structured-output chains.

What you will do:

- inspect the schema used by the chain
- inspect the prompt that receives the schema instructions
- forge the complete structured model
- optionally run a live model call if local credentials and a reachable provider are available


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'agentic_forge').exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os

STRUCT_DIR = PROJECT_ROOT / 'examples' / 'structured_output'
MATERIALS_DIR = STRUCT_DIR / 'materials'
ASSETS_DIR = STRUCT_DIR / 'assets'

from agentic_forge import forge
from agentic_forge.chat_prompt import ForgedChatPrompt
from agentic_forge.structured_output import ForgedResponseSchema, ForgedStructuredModel


## Step 1: Inspect the response schema

The schema is the contract that the model is expected to satisfy. In this example it defines a scene summary plus a list of detected objects with normalized bounding boxes.


In [ ]:
schema: ForgedResponseSchema = forge(MATERIALS_DIR / 'response_schema.yaml')
format_instruction = schema.get_format_instruction(include_description=True)
print(format_instruction[:800])


## Step 2: Inspect the prompt

The prompt YAML includes `{format_instruction}`. When the structured model is forged, that placeholder is filled with the formatting instructions generated from the response schema.


In [ ]:
prompt: ForgedChatPrompt = forge(MATERIALS_DIR / 'chat_prompt.yaml')
messages = prompt.format_messages(
    user_input='Describe the table image and return bounding boxes.',
    format_instruction=format_instruction,
    image_path=str(ASSETS_DIR / 'sample_with_boxes.jpg'),
)

for msg in prompt.compact_for_display(messages):
    print(msg)


## Step 3: Forge the complete structured model

`materials/structured_model.yaml` references the LLM manager, picks a concrete `provider:model` selector, and registers two named chains: `detect` and `refine`. Inspecting the runtime object here is enough to understand the wiring even without making a live API call.


In [ ]:
structured_model: ForgedStructuredModel = forge(MATERIALS_DIR / 'structured_model.yaml')
print('Chains:', list(structured_model.list_chains()))
print(structured_model.get_format_instruction('detect')[:600])


## Step 4: Optional live call

Live model calls are disabled by default so the notebook remains runnable without secrets or network access. Set `AGENTIC_FORGE_RUN_LIVE_LLM=1` only when the provider materials and credentials are configured locally.


In [ ]:
if os.environ.get('AGENTIC_FORGE_RUN_LIVE_LLM') == '1':
    payload = {
        'user_input': 'Where is Waldo?',
        'image_path': str(ASSETS_DIR / 'waldo_test.png'),
    }
    result = structured_model.invoke('detect', payload)
    print(dict(result))
else:
    print('Live LLM call skipped. Set AGENTIC_FORGE_RUN_LIVE_LLM=1 after configuring your keys.')
